In [1]:
# %% [markdown]
# # 18 — Master Results Collation: every dissertation table
#
# ═══════════════════════════════════════════════════════════════════════════
# WHAT THIS DOES
# ═══════════════════════════════════════════════════════════════════════════
#
# Walks every results directory on Drive, DISCOVERS what actually exists
# (rather than assuming a complete grid), audits completeness, de-duplicates,
# and emits every table needed for the dissertation — as printed tables, as
# CSVs, and as LaTeX.
#
# ARMS DISCOVERED FROM DRIVE
#   dense_kan/        Dense KAN
#   sparse_kan/       Sparse KAN v1  (defective init, eps=0.1,  G=14, 70 trials)
#   sparse_kan_v2/    Sparse KAN v2  (fixed,          eps=1e-5, G=14,  0 trials)
#   sparse_kan_v3/    Sparse KAN v3  (fixed,          eps=1e-5, G=14, 16 trials)
#   sparse_kan_v4/    Sparse KAN v4  (fixed, eps=1e-5, G=3 L0 / G=14 L1-2, 70)
#   dense_mlp/        Dense MLP
#   sparse_mlp/       Sparse MLP v1  (defective init)
#   sparse_mlp_v2/    Sparse MLP v2  (fixed init)
#
# Ridge, Polymodel and the volatility baseline live outside this tree; set
# BASELINE_DIRS below, or use the manual fallback (clearly flagged) until
# their real paths are wired in.
#
# ═══════════════════════════════════════════════════════════════════════════
# ROBUSTNESS NOTES (each learned from a specific earlier failure)
# ═══════════════════════════════════════════════════════════════════════════
#   - Metric values are sometimes stored as STRINGS (r2 came back as
#     '-0.117046475'). Every numeric read goes through safe_float().
#   - v3 has one duplicated config (afm/Split_B/binary/seed42 appears twice).
#     Duplicates are detected, reported, and resolved keeping the LAST row.
#   - Arms have DIFFERENT completeness. Nothing assumes 48 configs; the
#     audit runs first and every table reports its own n.
#   - Sortino is degenerate when average exposure is tiny (a Sortino of
#     11.48 at 0.2% invested). Backtest tables flag these rather than
#     quoting them.
#   - Nothing here re-runs a model. This is pure collation.

# %%
# ── COLAB SETUP ──
from google.colab import drive
drive.mount("/content/drive")
import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")

# %%
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

BASE = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2")
OUT_DIR   = BASE / "final_tables"
CSV_DIR   = OUT_DIR / "csv"
TEX_DIR   = OUT_DIR / "latex"
for d in (CSV_DIR, TEX_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Arm registry: directory -> display label + metadata for the tables ──
ARMS = {
    "dense_kan":     {"label": "Dense KAN",      "family": "KAN", "structure": "dense",
                      "note": "G=14, 70 trials"},
    "sparse_kan":    {"label": "Sparse KAN v1",  "family": "KAN", "structure": "sparse",
                      "note": "defective init, eps=0.1, G=14, 70 trials"},
    "sparse_kan_v2": {"label": "Sparse KAN v2",  "family": "KAN", "structure": "sparse",
                      "note": "fixed, eps=1e-5, G=14, 0 trials"},
    "sparse_kan_v3": {"label": "Sparse KAN v3",  "family": "KAN", "structure": "sparse",
                      "note": "fixed, eps=1e-5, G=14, 16 trials"},
    "sparse_kan_v4": {"label": "Sparse KAN v4",  "family": "KAN", "structure": "sparse",
                      "note": "fixed, eps=1e-5, G=3 L0, 70 trials"},
    "dense_mlp":     {"label": "Dense MLP",      "family": "MLP", "structure": "dense",
                      "note": "unaffected by init defect"},
    "sparse_mlp":    {"label": "Sparse MLP v1",  "family": "MLP", "structure": "sparse",
                      "note": "defective init"},
    "sparse_mlp_v2": {"label": "Sparse MLP v2",  "family": "MLP", "structure": "sparse",
                      "note": "fixed init, 0 trials"},
}

# Order used in every table
ARM_ORDER = ["dense_kan", "sparse_kan", "sparse_kan_v2", "sparse_kan_v3",
             "sparse_kan_v4", "dense_mlp", "sparse_mlp", "sparse_mlp_v2"]

DATASETS = ["agg_full_moments", "agg_means"]
SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]
TARGETS  = ["binary", "continuous"]

# ── Baselines. Set these paths if the results live elsewhere on Drive. ──
BASELINE_DIRS = {
    # "ridge":     Path("/content/drive/MyDrive/Thesis/Data/Results/Ridge"),
    # "polymodel": Path("/content/drive/MyDrive/Thesis/Data/Results/Polymodel"),
    # "vol_baseline": Path("/content/drive/MyDrive/Thesis/Data/Results/Vol_Baseline"),
}

# Manual fallback, used only if BASELINE_DIRS is empty or paths are missing.
# *** THESE MUST BE VERIFIED AGAINST THE ACTUAL BASELINE RESULT FILES BEFORE
# *** THEY GO IN THE THESIS. They are carried forward from earlier analysis
# *** notes, not read from disk by this notebook.
BASELINE_MANUAL = pd.DataFrame([
    {"arm": "vol_baseline", "label": "Volatility baseline (L3)", "family": "Baseline",
     "dataset": "agg_full_moments", "mse": 1.071, "verified": False},
    {"arm": "ridge", "label": "Ridge", "family": "Baseline",
     "dataset": "agg_full_moments", "mse": 1.176, "verified": False},
    {"arm": "ridge", "label": "Ridge", "family": "Baseline",
     "dataset": "agg_means", "mse": 1.090, "verified": False},
    {"arm": "polymodel", "label": "Polymodel (ava)", "family": "Baseline",
     "dataset": "agg_full_moments", "mse": 1.157, "verified": False},
    {"arm": "polymodel", "label": "Polymodel (ava)", "family": "Baseline",
     "dataset": "agg_means", "mse": 1.125, "verified": False},
])


def safe_float(v):
    """Metric values are sometimes stored as strings. Coerce or return NaN."""
    if v is None:
        return np.nan
    if isinstance(v, (int, float, np.floating, np.integer)):
        return float(v)
    try:
        return float(str(v).strip())
    except (TypeError, ValueError):
        return np.nan


print(f"Output directory: {OUT_DIR}")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## PART 1 — Discovery and completeness audit
# ═══════════════════════════════════════════════════════════════════════════
# Nothing assumes a complete 48-config grid. This walks the tree and reports
# exactly what exists per arm, so every downstream table can state its own n.

# %%
def parse_result_filename(stem):
    """
    Filenames look like:
        sparse_kan_agg_full_moments_continuous_Split_C_val
        dense_mlp_agg_means_binary_Split_A_test
    Returns (model_name, target_type, split_name, part) or None.
    Parsing anchors on the target-type token, which is unambiguous.
    """
    toks = stem.split("_")
    tgt_i = None
    for i, t in enumerate(toks):
        if t in ("binary", "continuous"):
            tgt_i = i
            break
    if tgt_i is None or tgt_i + 2 >= len(toks):
        return None
    model_name  = "_".join(toks[:tgt_i])
    target_type = toks[tgt_i]
    # split name is "Split" + "_" + letter
    if toks[tgt_i + 1] != "Split":
        return None
    split_name = f"{toks[tgt_i + 1]}_{toks[tgt_i + 2]}"
    part = "_".join(toks[tgt_i + 3:]) if tgt_i + 3 < len(toks) else None
    if part not in ("train", "val", "test"):
        return None
    return model_name, target_type, split_name, part


def discover_arm(arm_key):
    """Find every (seed, dataset, target, split, part) present for an arm."""
    arm_dir = BASE / arm_key
    if not arm_dir.exists():
        return []
    found = []
    for seed_dir in sorted(arm_dir.glob("seed_*")):
        try:
            seed = int(seed_dir.name.split("_")[1])
        except (IndexError, ValueError):
            continue
        pred_dir = seed_dir / "predictions"
        if not pred_dir.exists():
            continue
        for f in pred_dir.glob("*.parquet"):
            parsed = parse_result_filename(f.stem)
            if parsed is None:
                continue
            model_name, target_type, split_name, part = parsed
            # dataset is the model_name suffix after the architecture prefix
            dataset = None
            for ds in DATASETS:
                if model_name.endswith(ds):
                    dataset = ds
                    break
            if dataset is None:
                continue
            found.append({"arm": arm_key, "seed": seed, "dataset": dataset,
                          "target": target_type, "split": split_name,
                          "part": part, "model_name": model_name,
                          "seed_dir": seed_dir})
    return found


print("=" * 100)
print("PART 1: DISCOVERY AND COMPLETENESS AUDIT")
print("=" * 100)

inventory = []
for arm in ARM_ORDER:
    rows = discover_arm(arm)
    inventory += rows
    if not rows:
        print(f"  {arm:<16} NOT FOUND or empty")
        continue
    inv = pd.DataFrame(rows)
    test_only = inv[inv["part"] == "test"]
    n_cfg = len(test_only.drop_duplicates(["seed", "dataset", "target", "split"]))
    seeds = sorted(test_only["seed"].unique())
    print(f"  {arm:<16} {n_cfg:>3} test configs   seeds={seeds}   "
          f"datasets={sorted(test_only['dataset'].unique())}")

inv_df = pd.DataFrame(inventory)
inv_df.drop(columns=["seed_dir"]).to_csv(CSV_DIR / "00_inventory.csv", index=False)

# ── Explicit missing-config report against the full intended grid ──
print("\n" + "-" * 100)
print("MISSING CONFIGURATIONS (against the full seed x dataset x target x split grid)")
print("-" * 100)
missing_rows = []
for arm in ARM_ORDER:
    sub = inv_df[(inv_df.arm == arm) & (inv_df.part == "test")]
    if sub.empty:
        continue
    seeds_present = sorted(sub["seed"].unique())
    for sd in seeds_present:
        for ds in DATASETS:
            for tt in TARGETS:
                for sp in SPLITS:
                    hit = sub[(sub.seed == sd) & (sub.dataset == ds)
                              & (sub.target == tt) & (sub.split == sp)]
                    if hit.empty:
                        missing_rows.append({"arm": arm, "seed": sd, "dataset": ds,
                                             "target": tt, "split": sp})
if missing_rows:
    miss_df = pd.DataFrame(missing_rows)
    print(miss_df.groupby("arm").size().rename("n_missing").to_string())
    print("\nDetail:")
    print(miss_df.to_string(index=False))
    miss_df.to_csv(CSV_DIR / "00_missing_configs.csv", index=False)
else:
    print("  None — every arm is complete over the seeds it has.")
    miss_df = pd.DataFrame()


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## PART 2 — Build the master metrics frame
# ═══════════════════════════════════════════════════════════════════════════

# %%
from evaluation import load_predictions

print("\n" + "=" * 100)
print("PART 2: LOADING METRICS")
print("=" * 100)

METRIC_KEYS = ["mse", "r2", "mae", "auc", "derived_auc", "ece", "pred_std",
               "n_samples", "mean_actual", "mean_predicted", "derived_base_rate"]

records = []
targets_to_load = inv_df[inv_df["part"].isin(["train", "test"])] if len(inv_df) else pd.DataFrame()

for _, r in targets_to_load.iterrows():
    try:
        L = load_predictions(model_name=r["model_name"], split_name=r["split"],
                             target_type=r["target"], part=r["part"],
                             results_dir=r["seed_dir"])
    except Exception:
        continue
    md = L.get("metrics", {}) or {}
    rec = {"arm": r["arm"], "seed": r["seed"], "dataset": r["dataset"],
           "target": r["target"], "split": r["split"], "part": r["part"]}
    for k in METRIC_KEYS:
        if k in md:
            rec[k] = safe_float(md[k])
    hp = md.get("hyperparameters", {}) or {}
    for k in ["used_l1", "lr", "weight_decay", "batch_size", "reg_weight", "huber_delta"]:
        if k in hp:
            rec[k] = hp[k] if k == "used_l1" else safe_float(hp[k])
    records.append(rec)

master = pd.DataFrame(records)
print(f"Loaded {len(master)} metric records.")

# ── De-duplicate (v3 is known to have one duplicated config) ──
key = ["arm", "seed", "dataset", "target", "split", "part"]
dupe_mask = master.duplicated(subset=key, keep=False)
if dupe_mask.any():
    print(f"\n  WARNING: {int(dupe_mask.sum())} rows in "
          f"{master[dupe_mask].drop_duplicates(key).shape[0]} duplicated groups:")
    print(master[dupe_mask].sort_values(key)[key + ["mse", "r2", "auc"]]
          .to_string(index=False))
    print("  -> keeping LAST occurrence of each.")
    master = master.drop_duplicates(subset=key, keep="last")

master["label"] = master["arm"].map(lambda a: ARMS.get(a, {}).get("label", a))
master["family"] = master["arm"].map(lambda a: ARMS.get(a, {}).get("family", "?"))
master.to_csv(CSV_DIR / "01_master_metrics.csv", index=False)

test = master[master["part"] == "test"].copy()
train = master[master["part"] == "train"].copy()
print(f"\nTest records: {len(test)}   Train records: {len(train)}")


# ── LaTeX helper used by every table below ──
def to_latex(df, name, caption, label, float_fmt="%.3f", note=None):
    body = df.to_latex(index=True, float_format=float_fmt, escape=True,
                       na_rep="--", bold_rows=False)
    tex = (
        "\\begin{table}[htbp]\n\\centering\n"
        f"\\caption{{{caption}}}\n\\label{{tab:{label}}}\n"
        "\\small\n" + body
    )
    if note:
        tex += f"\\vspace{{2mm}}\n\\begin{{minipage}}{{\\textwidth}}\\footnotesize {note}\\end{{minipage}}\n"
    tex += "\\end{table}\n"
    (TEX_DIR / f"{name}.tex").write_text(tex)
    return tex


def save_and_show(df, name, caption, label, float_fmt="%.3f", note=None):
    df.to_csv(CSV_DIR / f"{name}.csv")
    to_latex(df, name, caption, label, float_fmt, note)
    print(df.to_string())
    print(f"\n  -> {CSV_DIR/f'{name}.csv'}  |  {TEX_DIR/f'{name}.tex'}")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## TABLE 1 — Headline model comparison (the main results table)
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("TABLE 1: HEADLINE COMPARISON — mean test MSE (continuous) and AUC (binary)")
print("=" * 100)

cont = test[test["target"] == "continuous"]
binr = test[test["target"] == "binary"]

t1_parts = []
if "mse" in cont.columns:
    mse = cont.groupby(["arm", "dataset"])["mse"].mean().unstack("dataset")
    mse.columns = [f"MSE {c}" for c in mse.columns]
    t1_parts.append(mse)
if "r2" in cont.columns:
    r2 = cont.groupby(["arm", "dataset"])["r2"].mean().unstack("dataset")
    r2.columns = [f"R2 {c}" for c in r2.columns]
    t1_parts.append(r2)
if "auc" in binr.columns and len(binr):
    auc = binr.groupby(["arm", "dataset"])["auc"].mean().unstack("dataset")
    auc.columns = [f"AUC {c}" for c in auc.columns]
    t1_parts.append(auc)

table1 = pd.concat(t1_parts, axis=1) if t1_parts else pd.DataFrame()
if len(table1):
    table1 = table1.reindex([a for a in ARM_ORDER if a in table1.index])
    table1.insert(0, "n_configs",
                  test.groupby("arm").size().reindex(table1.index))
    table1.index = [ARMS.get(a, {}).get("label", a) for a in table1.index]

    # Append baselines
    use_manual = not BASELINE_DIRS or not any(p.exists() for p in BASELINE_DIRS.values())
    if use_manual:
        print("\n  NOTE: baseline rows below come from the MANUAL fallback and are")
        print("        NOT read from disk. Verify against the baseline result files")
        print("        before these go in the thesis.\n")
        bl = (BASELINE_MANUAL.pivot_table(index="label", columns="dataset", values="mse")
              .rename(columns=lambda c: f"MSE {c}"))
        table1 = pd.concat([table1, bl], axis=0)

    save_and_show(
        table1, "table1_headline",
        "Out-of-sample performance, all models. MSE is in percent-squared on the "
        "continuous target; AUC is on the binary target. Values are means across "
        "four splits and all available seeds.",
        "headline",
        note="Baseline rows (volatility, Ridge, Polymodel) are carried from earlier "
             "analysis and must be verified against their source files. "
             "No neural architecture beats the four-parameter volatility regression.")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## TABLE 2 — Per-split MSE (shows which model wins where)
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("TABLE 2: PER-SPLIT TEST MSE, continuous target")
print("=" * 100)

for ds in DATASETS:
    sub = cont[cont["dataset"] == ds]
    if sub.empty or "mse" not in sub.columns:
        continue
    t2 = sub.groupby(["arm", "split"])["mse"].mean().unstack("split")
    t2 = t2.reindex([a for a in ARM_ORDER if a in t2.index])
    t2 = t2[[s for s in SPLITS if s in t2.columns]]
    t2["mean"] = t2.mean(axis=1)
    t2.index = [ARMS.get(a, {}).get("label", a) for a in t2.index]
    print(f"\n── {ds} ──")
    save_and_show(t2, f"table2_per_split_mse_{ds}",
                  f"Per-split test MSE (percent-squared), {ds.replace('_',' ')}. "
                  "Splits differ substantially in regime; Split B has the highest "
                  "target variance.",
                  f"persplit_{ds}")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## TABLE 3 — The capacity ladder (the central empirical result)
# ═══════════════════════════════════════════════════════════════════════════
# Layer-0 parameter counts read from the saved checkpoint model_config where
# available, so the x-axis of the ladder is exact rather than asserted.

# %%
print("\n" + "=" * 100)
print("TABLE 3: THE CAPACITY LADDER")
print("=" * 100)

import torch

def read_model_config(arm, dataset="agg_full_moments", seed=42):
    for tt in ["continuous", "binary"]:
        for sp in SPLITS:
            prefix = "sparse_kan" if "kan" in arm else "sparse_mlp"
            if arm.startswith("dense_kan"):
                prefix = "dense_kan"
            elif arm.startswith("dense_mlp"):
                prefix = "dense_mlp"
            p = (BASE / arm / f"seed_{seed}" / "checkpoints"
                 / f"{prefix}_{dataset}_{tt}_{sp}.pt")
            if p.exists():
                try:
                    ck = torch.load(p, map_location="cpu", weights_only=False)
                    return ck.get("model_config", {})
                except Exception:
                    continue
    return {}

ladder_rows = []
for arm in ["sparse_kan_v2", "sparse_kan_v3", "sparse_kan_v4", "sparse_mlp_v2"]:
    mc = read_model_config(arm)
    g0 = mc.get("grid_size_0", mc.get("grid_size"))
    n_basis0 = (g0 + mc.get("spline_order", 3)) if g0 is not None else None
    # layer-0 params: active edges x (basis + scaler + base). MLP has 1 weight/edge.
    if "kan" in arm and n_basis0 is not None:
        l0_params = 1699 * (n_basis0 + 2)
        per_edge = n_basis0 + 2
    else:
        l0_params = 1699 + 331     # weights + biases
        per_edge = 1699 / 1699     # 1 weight per edge
    sub = cont[(cont.arm == arm)]
    ladder_rows.append({
        "model": ARMS[arm]["label"],
        "L0 grid size": g0 if "kan" in arm else "--",
        "L0 params/edge": round(per_edge, 2) if isinstance(per_edge, float) else per_edge,
        "L0 total params": l0_params,
        "Total active params": mc.get("active_parameters", np.nan),
        "Search trials": {"sparse_kan_v2": 0, "sparse_kan_v3": 16,
                          "sparse_kan_v4": 70, "sparse_mlp_v2": 0}.get(arm, np.nan),
        "MSE afm": sub[sub.dataset == "agg_full_moments"]["mse"].mean()
                   if "mse" in sub.columns else np.nan,
        "MSE means": sub[sub.dataset == "agg_means"]["mse"].mean()
                     if "mse" in sub.columns else np.nan,
        "Mean R2": sub["r2"].mean() if "r2" in sub.columns else np.nan,
    })

table3 = pd.DataFrame(ladder_rows).set_index("model")
save_and_show(
    table3, "table3_capacity_ladder",
    "The capacity ladder. Identical structural sparsity, taxonomy, splits and "
    "seeds; only layer-0 per-edge capacity varies. Out-of-sample performance "
    "improves monotonically as layer-0 capacity falls.",
    "ladder",
    note="Layer-0 parameter counts assume 1,699 active layer-0 edges "
         "(agg\\_full\\_moments). Sparse MLP layer 0 is 1,699 weights plus 331 "
         "biases. The best-performing model used zero hyperparameter search trials.")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## TABLE 4 — Four-arm Sparse KAN comparison
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("TABLE 4: SPARSE KAN, FOUR ARMS")
print("=" * 100)

kan_arms = ["sparse_kan", "sparse_kan_v2", "sparse_kan_v3", "sparse_kan_v4"]
rows4 = []
for arm in kan_arms:
    c = cont[cont.arm == arm]
    b = binr[binr.arm == arm]
    # seed std computed per (dataset, split) then averaged
    seed_std_auc = (b.groupby(["dataset", "split"])["auc"].std().mean()
                    if "auc" in b.columns and len(b) else np.nan)
    seed_std_r2 = (c.groupby(["dataset", "split"])["r2"].std().mean()
                   if "r2" in c.columns and len(c) else np.nan)
    rows4.append({
        "arm": ARMS[arm]["label"],
        "configuration": ARMS[arm]["note"],
        "n test configs": len(c) + len(b),
        "MSE (mean)": c["mse"].mean() if "mse" in c.columns else np.nan,
        "R2 (mean)": c["r2"].mean() if "r2" in c.columns else np.nan,
        "AUC (mean)": b["auc"].mean() if "auc" in b.columns else np.nan,
        "seed sd (AUC)": seed_std_auc,
        "seed sd (R2)": seed_std_r2,
    })
table4 = pd.DataFrame(rows4).set_index("arm")
save_and_show(
    table4, "table4_four_arms",
    "Sparse KAN across four configurations. v1 to v2 is the clean controlled "
    "comparison: identical hyperparameters and seeds, only the initialisation "
    "and normalisation epsilon changed.",
    "fourarms",
    note="v3 received 16 search trials against v1's and v4's 70; any v3 shortfall "
         "is confounded with budget. No fixed-code arm at G=14 with 70 trials exists.")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## TABLE 5 — Sparse KAN vs Sparse MLP, head to head
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("TABLE 5: SPARSE MLP vs SPARSE KAN, PER CONFIGURATION")
print("=" * 100)

def cell_means(arm, target, metric):
    s = test[(test.arm == arm) & (test.target == target)]
    if metric not in s.columns or s.empty:
        return pd.Series(dtype=float)
    return s.groupby(["dataset", "split"])[metric].mean()

for kan_arm in ["sparse_kan_v2", "sparse_kan_v4"]:
    for tgt, metric, better in [("continuous", "r2", "higher"),
                                ("binary", "auc", "higher")]:
        a = cell_means(kan_arm, tgt, metric)
        b = cell_means("sparse_mlp_v2", tgt, metric)
        if a.empty or b.empty:
            continue
        comp = pd.DataFrame({ARMS[kan_arm]["label"]: a,
                             ARMS["sparse_mlp_v2"]["label"]: b})
        comp["MLP wins"] = comp.iloc[:, 1] > comp.iloc[:, 0]
        wins = int(comp["MLP wins"].sum())
        name = f"table5_headtohead_{kan_arm}_{tgt}"
        print(f"\n── {ARMS[kan_arm]['label']} vs Sparse MLP v2, {tgt} ({metric}) — "
              f"MLP wins {wins}/{len(comp)} ──")
        save_and_show(
            comp, name,
            f"{ARMS[kan_arm]['label']} versus Sparse MLP v2 on the {tgt} target "
            f"({metric}). Matched structural sparsity and taxonomy.",
            f"h2h_{kan_arm}_{tgt}",
            note=f"Sparse MLP v2 wins {wins} of {len(comp)} configurations using "
                 "roughly 16 times fewer active parameters and zero hyperparameter "
                 "search trials.")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## TABLE 6 — Binary AUC and derived AUC per split
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("TABLE 6: BINARY TEST AUC (mean +- seed sd), per split")
print("=" * 100)

for ds in DATASETS:
    sub = binr[binr.dataset == ds]
    if sub.empty or "auc" not in sub.columns:
        continue
    mean = sub.groupby(["arm", "split"])["auc"].mean().unstack("split")
    sd   = sub.groupby(["arm", "split"])["auc"].std().unstack("split")
    combo = mean.round(3).astype(str) + " (" + sd.round(3).astype(str) + ")"
    combo = combo.reindex([a for a in ARM_ORDER if a in combo.index])
    combo = combo[[s for s in SPLITS if s in combo.columns]]
    combo.index = [ARMS.get(a, {}).get("label", a) for a in combo.index]
    print(f"\n── {ds} ──")
    save_and_show(combo, f"table6_auc_{ds}",
                  f"Binary test AUC, mean (seed standard deviation), {ds.replace('_',' ')}.",
                  f"auc_{ds}", float_fmt="%s")

if "derived_auc" in cont.columns:
    print("\n── Derived AUC (continuous predictions ranked against the binary label) ──")
    t6b = cont.groupby(["arm", "dataset"])["derived_auc"].mean().unstack("dataset")
    t6b = t6b.reindex([a for a in ARM_ORDER if a in t6b.index])
    t6b.index = [ARMS.get(a, {}).get("label", a) for a in t6b.index]
    save_and_show(t6b, "table6b_derived_auc",
                  "Derived AUC: continuous predictions ranked against the binary "
                  "crash label. Measures ordering quality independent of calibration.",
                  "derivedauc")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## TABLE 7 — Overfitting: train vs test gap
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("TABLE 7: TRAIN-TEST GAP (evidence on capacity and overfitting)")
print("=" * 100)

gap_rows = []
for arm in ARM_ORDER:
    tr_b = train[(train.arm == arm) & (train.target == "binary")]
    te_b = test[(test.arm == arm) & (test.target == "binary")]
    tr_c = train[(train.arm == arm) & (train.target == "continuous")]
    te_c = test[(test.arm == arm) & (test.target == "continuous")]
    if tr_b.empty and tr_c.empty:
        continue
    gap_rows.append({
        "model": ARMS.get(arm, {}).get("label", arm),
        "train AUC": tr_b["auc"].mean() if "auc" in tr_b.columns else np.nan,
        "test AUC": te_b["auc"].mean() if "auc" in te_b.columns else np.nan,
        "AUC gap": (tr_b["auc"].mean() - te_b["auc"].mean())
                   if "auc" in tr_b.columns and "auc" in te_b.columns else np.nan,
        "train R2": tr_c["r2"].mean() if "r2" in tr_c.columns else np.nan,
        "test R2": te_c["r2"].mean() if "r2" in te_c.columns else np.nan,
        "R2 gap": (tr_c["r2"].mean() - te_c["r2"].mean())
                  if "r2" in tr_c.columns and "r2" in te_c.columns else np.nan,
    })
table7 = pd.DataFrame(gap_rows).set_index("model")
save_and_show(table7, "table7_train_test_gap",
              "Train versus test performance. A widening gap across the Sparse "
              "KAN arms accompanies the restoration of layer-0 capacity.",
              "gap")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## TABLE 8 — Backtests, with degenerate cells flagged
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("TABLE 8: BACKTEST SUMMARY")
print("=" * 100)

bt_rows = []
for arm in ARM_ORDER:
    p = BASE / arm / "backtests" / "backtest_summary.csv"
    if not p.exists():
        continue
    df = pd.read_csv(p)
    df["arm"] = arm
    bt_rows.append(df)

if bt_rows:
    bt = pd.concat(bt_rows, ignore_index=True)
    bt["label"] = bt["arm"].map(lambda a: ARMS.get(a, {}).get("label", a))
    # Flag degenerate Sortino: tiny exposure means almost no down days
    if "avg_exposure" in bt.columns:
        bt["sortino_degenerate"] = bt["avg_exposure"] < 0.05
        n_deg = int(bt["sortino_degenerate"].sum())
        print(f"  {n_deg} of {len(bt)} strategy cells have average exposure below 5%. "
              "Their Sortino figures are degenerate and are suppressed below.")
        bt.loc[bt["sortino_degenerate"], "sortino"] = np.nan
    bt.to_csv(CSV_DIR / "08_backtests_raw.csv", index=False)

    t8 = (bt.groupby(["label", "strategy"])[["sharpe", "annual_return", "max_drawdown"]]
          .mean().unstack("strategy"))
    save_and_show(t8, "table8_backtests",
                  "Backtest summary, seed-averaged signals, 3bps cost.",
                  "backtests",
                  note="With approximately 500 test days the standard error on an "
                       "annualised Sharpe ratio is roughly 0.7--0.8, so essentially "
                       "no difference in this table is statistically significant. "
                       "Sortino is suppressed wherever average exposure falls below "
                       "5 percent, since those cells contain almost no down days.")
    if "buy_hold_sharpe" in bt.columns:
        print("\nBuy-and-hold Sharpe by split (for reference):")
        print(bt.groupby("split")["buy_hold_sharpe"].mean().round(3).to_string())
else:
    print("  No backtest_summary.csv found in any arm directory.")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## TABLE 9 — Hyperparameter selection patterns
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("TABLE 9: HYPERPARAMETER SELECTION")
print("=" * 100)

if "used_l1" in test.columns and test["used_l1"].notna().any():
    hp_rows = []
    for arm in ARM_ORDER:
        s = test[test.arm == arm]
        if s.empty:
            continue
        rw = s["reg_weight"].dropna() if "reg_weight" in s.columns else pd.Series(dtype=float)
        hp_rows.append({
            "model": ARMS.get(arm, {}).get("label", arm),
            "n": len(s),
            "frac with-L1": s["used_l1"].mean() if s["used_l1"].notna().any() else np.nan,
            "median log10(reg_weight)": np.log10(rw[rw > 0]).median() if len(rw[rw > 0]) else np.nan,
            "median lr": s["lr"].median() if "lr" in s.columns else np.nan,
        })
    table9 = pd.DataFrame(hp_rows).set_index("model")
    save_and_show(table9, "table9_hyperparameters",
                  "Hyperparameter selection patterns across arms.",
                  "hyperparams",
                  note="A with-L1 fraction near 0.5 indicates the L1 penalty was not "
                       "reliably preferred over its absence, which is weak evidence "
                       "that regularisation strength is not the binding constraint.")
else:
    print("  No hyperparameter metadata found in the saved test metrics.")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## FINAL — Index of everything produced
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("OUTPUT INDEX")
print("=" * 100)
print(f"\nCSV  ({CSV_DIR}):")
for f in sorted(CSV_DIR.glob("*.csv")):
    print(f"  {f.name}")
print(f"\nLaTeX ({TEX_DIR}):")
for f in sorted(TEX_DIR.glob("*.tex")):
    print(f"  {f.name}")

print("\n" + "=" * 100)
print("CAVEATS TO CARRY INTO THE WRITE-UP")
print("=" * 100)
print("""
  1. Baseline rows (volatility, Ridge, Polymodel) in Table 1 come from the
     manual fallback unless BASELINE_DIRS was configured. Verify them.
  2. Arms differ in completeness -- every table reports its own n. v3 in
     particular had missing studies and one duplicated configuration.
  3. v3 received 16 search trials against v1's and v4's 70. No fixed-code
     arm at G=14 with 70 trials exists, so the v4-versus-v3 comparison is
     confounded with budget.
  4. Backtest differences are not statistically significant at this sample
     size; Sortino is suppressed where exposure is below 5%.
  5. Seed standard deviations are computed per (dataset, split) and then
     averaged, which is the correct order -- pooling first would understate
     them.
""")

Mounted at /content/drive
Output directory: /content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/final_tables
PART 1: DISCOVERY AND COMPLETENESS AUDIT
  dense_kan         48 test configs   seeds=[np.int64(42), np.int64(123), np.int64(456)]   datasets=['agg_full_moments', 'agg_means']
  sparse_kan        48 test configs   seeds=[np.int64(42), np.int64(123), np.int64(456)]   datasets=['agg_full_moments', 'agg_means']
  sparse_kan_v2     48 test configs   seeds=[np.int64(42), np.int64(123), np.int64(456)]   datasets=['agg_full_moments', 'agg_means']
  sparse_kan_v3     48 test configs   seeds=[np.int64(42), np.int64(123), np.int64(456)]   datasets=['agg_full_moments', 'agg_means']
  sparse_kan_v4     48 test configs   seeds=[np.int64(42), np.int64(123), np.int64(456)]   datasets=['agg_full_moments', 'agg_means']
  dense_mlp         48 test configs   seeds=[np.int64(42), np.int64(123), np.int64(456)]   datasets=['agg_full_moments', 'agg_means']
  sparse_mlp        48 test conf

In [1]:
# %% [markdown]
# # 18b — Master Results Collation: Ridge / Polymodel / Volatility Baseline
#
# ═══════════════════════════════════════════════════════════════════════════
# LOCAL, CPU-ONLY VARIANT OF NOTEBOOK 18
# ═══════════════════════════════════════════════════════════════════════════
#
# Differences from the Colab master collation (notebook 18):
#
#   1. NO SEED SUBFOLDER. Ridge and Polymodel are both deterministic single
#      runs (Ridge's grid search and Polymodel's stratified-fold LNLM fit use
#      no randomness that varies run-to-run), so their predictions/ and
#      metrics/ sit DIRECTLY under RESULTS_DIR, not under RESULTS_DIR/seed_*/.
#      discover_local_arm() below reflects that -- it does not look for
#      seed_* directories at all.
#
#   2. NO TORCH, NO CHECKPOINTS. There is no capacity-ladder table here --
#      that concept (layer-0 parameter count vs performance) only applies to
#      the KAN/MLP architectures. This script never imports torch and never
#      touches a GPU.
#
#   3. BACKTESTS ARE RECOMPUTED, NOT READ. Neither the Ridge nor the
#      Polymodel notebook persisted backtest_summary.csv to disk -- they only
#      printed run_full_backtest()'s output inline. This script re-runs
#      run_full_backtest() directly on the saved prediction parquets (cheap:
#      pure numpy on a few thousand rows) rather than skipping the table.
#
#   4. POLYMODEL HAS FOUR VARIANTS PER DATASET (equal / rmse / ava / ava_unc),
#      encoded in the model_name itself (e.g. "polymodel_ava_agg_means").
#      Each variant is treated as its own "arm" in the tables below, with AVA
#      flagged as the reported/primary variant (matching the original
#      notebook's choice to backtest AVA only).
#
#   5. VOLATILITY BASELINE PATH IS A GUESS. No volatility-baseline notebook
#      was available when this script was written. ROOT/"Vol_Baseline" is
#      assumed; if that directory or its predictions/ subfolder doesn't
#      exist, this script prints a clear "NOT FOUND" line and continues --
#      it does NOT silently fall back to fabricated numbers the way the
#      manual BASELINE_MANUAL table in notebook 18 did.
#
# Run this from the Code/ directory (or wherever evaluation.py lives) so the
# `from evaluation import ...` import below resolves the same way it did in
# the original Ridge/Polymodel notebooks.

# %%
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from evaluation import load_predictions, run_full_backtest

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION -- adjust ROOT for your local machine
# ═══════════════════════════════════════════════════════════════════════════

# NOTE: this is "Dense_vs_Sparse_KAN", NOT "..._v2" -- Ridge and Polymodel
# were written to save under the non-"_v2" results tree (see their
# RESULTS_DIR constants), unlike the neural architectures collated in
# notebook 18.
ROOT = Path("../../..") / "Data" / "Results" / "Dense_vs_Sparse_KAN"

ARM_DIRS = {
    "ridge":     ROOT / "Ridge",
    "polymodel": ROOT / "Polymodel",
    "vol_baseline": ROOT / "Vol_Baseline",   # <-- UNVERIFIED, adjust if wrong
}

OUT_DIR = ROOT / "final_tables_baselines"
CSV_DIR = OUT_DIR / "csv"
TEX_DIR = OUT_DIR / "latex"
for d in (CSV_DIR, TEX_DIR):
    d.mkdir(parents=True, exist_ok=True)

DATASETS = ["agg_full_moments", "agg_means"]
SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]
TARGETS  = ["binary", "continuous"]

POLYMODEL_VARIANTS = ["equal", "rmse", "ava", "ava_unc"]
POLYMODEL_PRIMARY  = "ava"   # matches the original notebook's backtest choice

LABELS = {
    "ridge":              "Ridge",
    "polymodel_equal":    "Polymodel (equal)",
    "polymodel_rmse":     "Polymodel (RMSE-weighted)",
    "polymodel_ava":      "Polymodel (AVA)",
    "polymodel_ava_unc":  "Polymodel (AVA + uncertainty)",
    "vol_baseline":       "Volatility baseline",
}
ARM_ORDER = ["vol_baseline", "ridge", "polymodel_equal", "polymodel_rmse",
             "polymodel_ava", "polymodel_ava_unc"]


def safe_float(v):
    """Metric values are sometimes stored as strings -- coerce or NaN."""
    if v is None:
        return np.nan
    if isinstance(v, (int, float, np.floating, np.integer)):
        return float(v)
    try:
        return float(str(v).strip())
    except (TypeError, ValueError):
        return np.nan


print(f"Output directory: {OUT_DIR}")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## PART 1 — Discovery (flat structure, no seed_* subfolders)
# ═══════════════════════════════════════════════════════════════════════════

# %%
def parse_result_filename(stem):
    """
    Filenames look like:
        ridge_agg_full_moments_continuous_Split_C_val
        polymodel_ava_agg_means_binary_Split_A_test
    Returns (model_name, target_type, split_name, part) or None.
    Identical logic to notebook 18 -- parsing anchors on the target-type
    token, which is unambiguous regardless of how many underscores the
    architecture prefix contains.
    """
    toks = stem.split("_")
    tgt_i = None
    for i, t in enumerate(toks):
        if t in ("binary", "continuous"):
            tgt_i = i
            break
    if tgt_i is None or tgt_i + 2 >= len(toks):
        return None
    model_name  = "_".join(toks[:tgt_i])
    target_type = toks[tgt_i]
    if toks[tgt_i + 1] != "Split":
        return None
    split_name = f"{toks[tgt_i + 1]}_{toks[tgt_i + 2]}"
    part = "_".join(toks[tgt_i + 3:]) if tgt_i + 3 < len(toks) else None
    if part not in ("train", "val", "test"):
        return None
    return model_name, target_type, split_name, part


def dataset_from_model_name(model_name):
    for ds in DATASETS:
        if model_name.endswith(ds):
            return ds
    return None


def prefix_and_variant(model_name, dataset, source):
    """
    Strips the trailing "_{dataset}" to get the architecture prefix, then
    (for polymodel only) splits out the variant name.
        ridge_agg_full_moments          -> prefix="ridge",             variant=None
        polymodel_ava_agg_means         -> prefix="polymodel_ava",     variant="ava"
        vol_baseline_agg_full_moments   -> prefix="vol_baseline",      variant=None
    """
    prefix = model_name[: -(len(dataset) + 1)]
    variant = None
    if source == "polymodel" and prefix.startswith("polymodel_"):
        candidate = prefix[len("polymodel_"):]
        if candidate in POLYMODEL_VARIANTS:
            variant = candidate
    return prefix, variant


def discover_local_arm(source, arm_dir):
    """
    Flat-structure discovery: predictions live directly in
    arm_dir/predictions/*.parquet, with NO seed_* level above them.
    """
    arm_dir = Path(arm_dir)
    pred_dir = arm_dir / "predictions"
    if not pred_dir.exists():
        print(f"  {source:<14} NOT FOUND at {pred_dir}")
        return []

    found = []
    for f in pred_dir.glob("*.parquet"):
        parsed = parse_result_filename(f.stem)
        if parsed is None:
            continue
        model_name, target_type, split_name, part = parsed
        dataset = dataset_from_model_name(model_name)
        if dataset is None:
            continue
        prefix, variant = prefix_and_variant(model_name, dataset, source)
        found.append({
            "source": source, "prefix": prefix, "variant": variant,
            "dataset": dataset, "target": target_type, "split": split_name,
            "part": part, "model_name": model_name, "results_dir": arm_dir,
        })
    return found


print("=" * 100)
print("PART 1: DISCOVERY (flat structure -- no seed_* level)")
print("=" * 100)

inventory = []
for source, arm_dir in ARM_DIRS.items():
    rows = discover_local_arm(source, arm_dir)
    inventory += rows
    if not rows:
        continue
    inv = pd.DataFrame(rows)
    test_only = inv[inv["part"] == "test"]
    n_cfg = len(test_only.drop_duplicates(["prefix", "dataset", "target", "split"]))
    prefixes = sorted(test_only["prefix"].unique())
    print(f"  {source:<14} {n_cfg:>3} test configs   prefixes={prefixes}   "
          f"datasets={sorted(test_only['dataset'].unique())}")

inv_df = pd.DataFrame(inventory)
if len(inv_df):
    inv_df.drop(columns=["results_dir"]).to_csv(CSV_DIR / "00_inventory.csv", index=False)

# ── Explicit missing-config report against the full intended grid ──
print("\n" + "-" * 100)
print("MISSING CONFIGURATIONS (against the full dataset x target x split grid)")
print("-" * 100)
missing_rows = []
for prefix in inv_df["prefix"].unique() if len(inv_df) else []:
    sub = inv_df[(inv_df.prefix == prefix) & (inv_df.part == "test")]
    for ds in DATASETS:
        for tt in TARGETS:
            for sp in SPLITS:
                hit = sub[(sub.dataset == ds) & (sub.target == tt) & (sub.split == sp)]
                if hit.empty:
                    missing_rows.append({"prefix": prefix, "dataset": ds,
                                         "target": tt, "split": sp})
if missing_rows:
    miss_df = pd.DataFrame(missing_rows)
    print(miss_df.groupby("prefix").size().rename("n_missing").to_string())
    miss_df.to_csv(CSV_DIR / "00_missing_configs.csv", index=False)
else:
    print("  None -- every discovered prefix is complete over the grid it has.")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## PART 2 — Build the master metrics frame
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("PART 2: LOADING METRICS")
print("=" * 100)

METRIC_KEYS = ["mse", "r2", "mae", "auc", "derived_auc", "ece", "pred_std",
               "n_samples", "mean_actual", "mean_predicted", "derived_base_rate",
               "brier"]

records = []
targets_to_load = inv_df[inv_df["part"].isin(["train", "test"])] if len(inv_df) else pd.DataFrame()

for _, r in targets_to_load.iterrows():
    try:
        L = load_predictions(model_name=r["model_name"], split_name=r["split"],
                             target_type=r["target"], part=r["part"],
                             results_dir=r["results_dir"])
    except Exception as e:
        print(f"  ! failed to load {r['model_name']}/{r['split']}/{r['target']}/"
              f"{r['part']}: {e}")
        continue
    md = L.get("metrics", {}) or {}
    rec = {"source": r["source"], "prefix": r["prefix"], "variant": r["variant"],
           "dataset": r["dataset"], "target": r["target"], "split": r["split"],
           "part": r["part"]}
    for k in METRIC_KEYS:
        if k in md:
            rec[k] = safe_float(md[k])
    hp = md.get("hyperparameters", {}) or {}
    for k in ["C", "alpha", "n_hermite", "n_folds", "n_mu",
              "believability_normalised"]:
        if k in hp:
            rec[k] = hp[k]
    records.append(rec)

master = pd.DataFrame(records)
print(f"Loaded {len(master)} metric records.")

# ── De-duplicate defensively (same key as notebook 18, no seed column here) ──
key = ["prefix", "dataset", "target", "split", "part"]
if len(master):
    dupe_mask = master.duplicated(subset=key, keep=False)
    if dupe_mask.any():
        print(f"\n  WARNING: {int(dupe_mask.sum())} duplicated rows found -- "
              "keeping LAST occurrence of each.")
        master = master.drop_duplicates(subset=key, keep="last")

master["label"] = master["prefix"].map(lambda p: LABELS.get(p, p))
master.to_csv(CSV_DIR / "01_master_metrics.csv", index=False)

test  = master[master["part"] == "test"].copy()
train = master[master["part"] == "train"].copy()
print(f"\nTest records: {len(test)}   Train records: {len(train)}")


def to_latex(df, name, caption, label, float_fmt="%.3f", note=None):
    body = df.to_latex(index=True, float_format=float_fmt, escape=True,
                       na_rep="--", bold_rows=False)
    tex = (
        "\\begin{table}[htbp]\n\\centering\n"
        f"\\caption{{{caption}}}\n\\label{{tab:{label}}}\n"
        "\\small\n" + body
    )
    if note:
        tex += f"\\vspace{{2mm}}\n\\begin{{minipage}}{{\\textwidth}}\\footnotesize {note}\\end{{minipage}}\n"
    tex += "\\end{table}\n"
    (TEX_DIR / f"{name}.tex").write_text(tex)
    return tex


def save_and_show(df, name, caption, label, float_fmt="%.3f", note=None):
    df.to_csv(CSV_DIR / f"{name}.csv")
    to_latex(df, name, caption, label, float_fmt, note)
    print(df.to_string())
    print(f"\n  -> {CSV_DIR/f'{name}.csv'}  |  {TEX_DIR/f'{name}.tex'}")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## TABLE 1 — Headline comparison (baselines only)
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("TABLE 1: HEADLINE COMPARISON -- mean test MSE (continuous) and AUC (binary)")
print("=" * 100)

cont = test[test["target"] == "continuous"]
binr = test[test["target"] == "binary"]

t1_parts = []
if "mse" in cont.columns and len(cont):
    mse = cont.groupby(["prefix", "dataset"])["mse"].mean().unstack("dataset")
    mse.columns = [f"MSE {c}" for c in mse.columns]
    t1_parts.append(mse)
if "r2" in cont.columns and len(cont):
    r2 = cont.groupby(["prefix", "dataset"])["r2"].mean().unstack("dataset")
    r2.columns = [f"R2 {c}" for c in r2.columns]
    t1_parts.append(r2)
if "derived_auc" in cont.columns and len(cont):
    dauc = cont.groupby(["prefix", "dataset"])["derived_auc"].mean().unstack("dataset")
    dauc.columns = [f"DerivedAUC {c}" for c in dauc.columns]
    t1_parts.append(dauc)
if "auc" in binr.columns and len(binr):
    auc = binr.groupby(["prefix", "dataset"])["auc"].mean().unstack("dataset")
    auc.columns = [f"AUC {c}" for c in auc.columns]
    t1_parts.append(auc)

table1 = pd.concat(t1_parts, axis=1) if t1_parts else pd.DataFrame()
if len(table1):
    table1 = table1.reindex([p for p in ARM_ORDER if p in table1.index])
    table1.insert(0, "n_configs",
                  test.groupby("prefix").size().reindex(table1.index))
    table1.index = [LABELS.get(p, p) for p in table1.index]

    save_and_show(
        table1, "table1_headline_baselines",
        "Out-of-sample performance, baseline models only (Ridge, Polymodel "
        "variants, volatility baseline). MSE is percent-squared on the "
        "continuous target; AUC is on the binary target. Values are means "
        "across the four splits.",
        "headline_baselines",
        note="Polymodel is reported under all four aggregation variants; AVA "
             "is the variant used for backtesting in the original notebook and "
             "should be treated as the primary comparison point against Ridge "
             "and the neural architectures.")
else:
    print("  No data available to build Table 1 -- check ARM_DIRS paths above.")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## TABLE 2 — Per-split metrics
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("TABLE 2: PER-SPLIT TEST MSE, continuous target")
print("=" * 100)

for ds in DATASETS:
    sub = cont[cont["dataset"] == ds]
    if sub.empty or "mse" not in sub.columns:
        continue
    t2 = sub.groupby(["prefix", "split"])["mse"].mean().unstack("split")
    t2 = t2.reindex([p for p in ARM_ORDER if p in t2.index])
    t2 = t2[[s for s in SPLITS if s in t2.columns]]
    t2["mean"] = t2.mean(axis=1)
    t2.index = [LABELS.get(p, p) for p in t2.index]
    print(f"\n-- {ds} --")
    save_and_show(t2, f"table2_per_split_mse_{ds}",
                  f"Per-split test MSE (percent-squared), baselines, {ds.replace('_',' ')}.",
                  f"persplit_baselines_{ds}")

print("\n" + "=" * 100)
print("TABLE 2b: PER-SPLIT TEST AUC, binary target")
print("=" * 100)

for ds in DATASETS:
    sub = binr[binr["dataset"] == ds]
    if sub.empty or "auc" not in sub.columns:
        continue
    t2b = sub.groupby(["prefix", "split"])["auc"].mean().unstack("split")
    t2b = t2b.reindex([p for p in ARM_ORDER if p in t2b.index])
    t2b = t2b[[s for s in SPLITS if s in t2b.columns]]
    t2b["mean"] = t2b.mean(axis=1)
    t2b.index = [LABELS.get(p, p) for p in t2b.index]
    print(f"\n-- {ds} --")
    save_and_show(t2b, f"table2b_per_split_auc_{ds}",
                  f"Per-split test AUC, baselines, {ds.replace('_',' ')}.",
                  f"persplit_auc_baselines_{ds}")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## TABLE 3 — Backtests (RECOMPUTED from saved predictions, CPU-only)
# ═══════════════════════════════════════════════════════════════════════════
# Neither the Ridge nor the Polymodel notebook persisted a backtest_summary
# csv, so this section re-runs run_full_backtest() directly on the val/test
# prediction parquets. This is pure numpy on a few thousand rows per config
# and takes seconds total on CPU -- nothing here needs a GPU.

# %%
print("\n" + "=" * 100)
print("TABLE 3: BACKTESTS (recomputed from saved predictions)")
print("=" * 100)

SIGNAL_COL = {"binary": "y_prob", "continuous": "y_pred"}
GO_CASH_WHEN = {"binary": "above", "continuous": "below"}

bt_rows = []
configs = inv_df[inv_df["part"] == "test"].drop_duplicates(
    ["prefix", "dataset", "target", "split"]
) if len(inv_df) else pd.DataFrame()

for _, cfg in configs.iterrows():
    prefix, dataset, target, split = (cfg["prefix"], cfg["dataset"],
                                      cfg["target"], cfg["split"])
    model_name = f"{prefix}_{dataset}"
    results_dir = cfg["results_dir"]
    sig_col = SIGNAL_COL[target]

    try:
        val = load_predictions(model_name=model_name, split_name=split,
                               target_type=target, part="val",
                               results_dir=results_dir)["predictions"]
        te = load_predictions(model_name=model_name, split_name=split,
                              target_type=target, part="test",
                              results_dir=results_dir)["predictions"]
    except Exception as e:
        print(f"  ! backtest skipped for {model_name}/{split}/{target}: {e}")
        continue

    if sig_col not in val.columns or sig_col not in te.columns:
        print(f"  ! backtest skipped for {model_name}/{split}/{target}: "
              f"missing column '{sig_col}'")
        continue

    bt = run_full_backtest(
        val_returns=val["daily_return"].values,
        val_signal=val[sig_col].values,
        test_returns=te["daily_return"].values,
        test_signal=te[sig_col].values,
        go_cash_when=GO_CASH_WHEN[target],
        model_name=model_name,
        split_name=split,
    )

    for strategy_name, strategy_key in [("simple", "simple"), ("risk_scaled", "risk_scaled")]:
        d = bt[strategy_key]
        bt_rows.append({
            "prefix": prefix, "dataset": dataset, "target": target, "split": split,
            "strategy": strategy_name,
            "sharpe": d["sharpe"], "sortino": d["sortino"],
            "annual_return": d["annual_return"], "max_drawdown": d["max_drawdown"],
            "avg_exposure": d["avg_exposure"],
            "buy_hold_sharpe": bt["buy_hold"]["sharpe"],
        })

if bt_rows:
    bt = pd.DataFrame(bt_rows)
    bt["label"] = bt["prefix"].map(lambda p: LABELS.get(p, p))

    # Same degenerate-Sortino guard as notebook 18: tiny exposure means
    # almost no down days, so Sortino is not a meaningful number there.
    bt["sortino_degenerate"] = bt["avg_exposure"] < 0.05
    n_deg = int(bt["sortino_degenerate"].sum())
    print(f"  {n_deg} of {len(bt)} strategy cells have average exposure below "
          "5%. Their Sortino figures are suppressed below.")
    bt.loc[bt["sortino_degenerate"], "sortino"] = np.nan

    bt.to_csv(CSV_DIR / "03_backtests_raw.csv", index=False)

    t3 = (bt.groupby(["label", "strategy"])[["sharpe", "annual_return", "max_drawdown"]]
          .mean().unstack("strategy"))
    save_and_show(t3, "table3_backtests_baselines",
                  "Backtest summary, baselines, recomputed from saved "
                  "predictions, 3bps cost, EMA span 5 (evaluation.py defaults).",
                  "backtests_baselines",
                  note="Recomputed directly from stored predictions since "
                       "neither source notebook persisted a backtest summary "
                       "to disk. Sortino is suppressed wherever average "
                       "exposure falls below 5 percent.")

    print("\nBuy-and-hold Sharpe by split (for reference):")
    print(bt.groupby("split")["buy_hold_sharpe"].mean().round(3).to_string())
else:
    print("  No backtests could be recomputed -- check that predictions "
        "parquets contain y_prob/y_pred and daily_return columns.")


# %% [markdown]
# ═══════════════════════════════════════════════════════════════════════════
# ## FINAL — Index of everything produced
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 100)
print("OUTPUT INDEX")
print("=" * 100)
print(f"\nCSV  ({CSV_DIR}):")
for f in sorted(CSV_DIR.glob("*.csv")):
    print(f"  {f.name}")
print(f"\nLaTeX ({TEX_DIR}):")
for f in sorted(TEX_DIR.glob("*.tex")):
    print(f"  {f.name}")

print("\n" + "=" * 100)
print("CAVEATS TO CARRY INTO THE WRITE-UP")
print("=" * 100)
print(f"""
  1. vol_baseline path ({ARM_DIRS['vol_baseline']}) is UNVERIFIED -- no
     volatility-baseline notebook was available when this script was
     written. If it printed "NOT FOUND" above, update ARM_DIRS to the real
     path and re-run.
  2. Ridge and Polymodel are single deterministic runs -- there is no seed
     variance to report for these two arms, unlike the neural architectures.
  3. Backtest figures here are RECOMPUTED from saved predictions, not read
     from a persisted summary -- verify evaluation.py's cost/EMA defaults
     (3bps, span 5) match what you want reported before quoting these.
  4. Polymodel AVA is the variant used for backtesting/headline comparison
     in the original notebook; equal/rmse/ava_unc are reported for
     completeness but AVA is the primary comparison point.
  5. With ~500 test days the standard error on an annualised Sharpe ratio
     is roughly 0.7-0.8 -- treat Table 3 differences as indicative, not
     statistically decisive.
""")

Output directory: ..\..\..\Data\Results\Dense_vs_Sparse_KAN\final_tables_baselines
PART 1: DISCOVERY (flat structure -- no seed_* level)
  ridge           16 test configs   prefixes=['ridge']   datasets=['agg_full_moments', 'agg_means']
  polymodel       32 test configs   prefixes=['polymodel_ava', 'polymodel_ava_unc', 'polymodel_equal', 'polymodel_rmse']   datasets=['agg_full_moments', 'agg_means']
  vol_baseline   NOT FOUND at ..\..\..\Data\Results\Dense_vs_Sparse_KAN\Vol_Baseline\predictions

----------------------------------------------------------------------------------------------------
MISSING CONFIGURATIONS (against the full dataset x target x split grid)
----------------------------------------------------------------------------------------------------
prefix
polymodel_ava        8
polymodel_ava_unc    8
polymodel_equal      8
polymodel_rmse       8

PART 2: LOADING METRICS
Loaded 96 metric records.

Test records: 48   Train records: 48

TABLE 1: HEADLINE COMPARISON -- mean